# House Prices Preprocessing (Bronze → Silver)

Cleans the UK Land Registry **Price Paid Data (PPD)** into a tidy,
transaction-level silver table for later crime-analytics enrichment.

- **Input:** `../data/bronze/price_paid/pp-*.csv` (headerless, 16-col standard PPD)
- **Output:** `../data/silver/houseprices_clean.csv`

Cleaning rules follow `documentation/houseprices_preprocess.md`: keep only
operational columns, residential property types, standard sales, and
realistic prices. Each file is cleaned **independently** and only the
cleaned per-file frames are concatenated (no early raw concat); the column
order is **date first, price last** per the project requirement.

**Out of scope for this iteration** (deferred to a later notebook): the
`postcode → lsoa_code` mapping and the `lsoa × month` aggregation. The raw
`postcode` column is kept as-is; no join and no aggregation are performed here.

In [20]:
import os
import glob
import pandas as pd

# **1. Bronze → Silver — load & clean each file**

The PPD CSVs have **no header row**, so the standard 16-column Land
Registry schema is assigned on read. Every file is cleaned on its own by
`prepare_price_paid` *before* anything is concatenated — this keeps peak
memory low (each file is narrowed 16→6 cols and filtered ~16% smaller)
and matches the per-file pattern used in `adi_preprocessing.ipynb`.

In [21]:
# Standard Land Registry Price Paid Data column order (raw, headerless)
PPD_COLUMNS = [
    "transaction_id", "price", "date_of_transfer", "postcode",
    "property_type", "old_new", "duration", "paon", "saon",
    "street", "locality", "town_city", "district", "county",
    "ppd_category_type", "record_status",
]

# Final operational column order: date first, price last.
# transaction_id is carried as a dedup helper and dropped after concat.
OUTPUT_COLUMNS = [
    "transaction_id", "date_of_transfer", "postcode",
    "property_type", "ppd_category_type", "price",
]

RESIDENTIAL = {"D", "S", "T", "F"}


def standardise_postcode(series):
    """Uppercase, strip, and collapse any internal whitespace run to a
    single space (ONS 'pcds' single-space format).
    """
    return (
        series.str.upper()
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )


def prepare_price_paid(df):
    """Clean ONE raw per-file PPD frame (per `houseprices_preprocess.md`).

    Steps: keep operational columns in final order, cast types,
    standardise postcode (drop blank), and apply the residential /
    standard-sale / minimum-price filters. Returns the cleaned frame.
    """
    df = df[OUTPUT_COLUMNS].copy()

    # Types: price -> numeric, date_of_transfer -> datetime
    # (source format is 'YYYY-MM-DD 00:00')
    df["price"] = pd.to_numeric(df["price"], errors="coerce")
    df["date_of_transfer"] = pd.to_datetime(df["date_of_transfer"], errors="coerce")

    # Postcode standardisation; drop blank/null (cannot be enriched later).
    # Kept as a column -- no LSOA join here.
    df["postcode"] = standardise_postcode(df["postcode"])
    df = df[df["postcode"].notna() & (df["postcode"] != "")]

    # Filters: residential only, standard sales only, realistic price only
    df = df[df["property_type"].isin(RESIDENTIAL)]
    df = df[df["ppd_category_type"] == "A"]
    df = df[df["price"] >= 50000]

    return df


def load_price_paid(pattern, prepare_fn):
    """Load each headerless PPD CSV matching `pattern` SEPARATELY, clean it
    with `prepare_fn`, and return (cleaned_frames, total_raw_rows).

    No raw multi-file frame is ever materialised: each file is read,
    cleaned, and only the cleaned slice is retained.
    """
    files = sorted(glob.glob(pattern))

    if not files:
        print(f"No files found for: {pattern}")
        return [], 0

    cleaned, total_raw = [], 0
    for file in files:
        df = pd.read_csv(file, header=None, names=PPD_COLUMNS, dtype=str)
        total_raw += len(df)
        df = prepare_fn(df)
        print(f"{os.path.basename(file)}: raw rows kept -> {df.shape}")
        cleaned.append(df)

    return cleaned, total_raw


cleaned_dfs, raw_row_count = load_price_paid(
    "../data/bronze/price_paid/pp-*.csv", prepare_price_paid
)
print("total raw rows:", raw_row_count)

pp-2017-part1.csv: raw rows kept -> (453017, 6)
pp-2017-part2.csv: raw rows kept -> (446323, 6)
pp-2018.csv: raw rows kept -> (869157, 6)
pp-2019.csv: raw rows kept -> (837430, 6)
pp-2020.csv: raw rows kept -> (748661, 6)
pp-2021.csv: raw rows kept -> (1082719, 6)
total raw rows: 5295467


In [22]:
# Concat ONLY the already-cleaned per-file frames
df = pd.concat(cleaned_dfs, ignore_index=True)
print("shape after cleaning + concat:", df.shape)
df.head()

shape after cleaning + concat: (4437307, 6)


,transaction_id,date_of_transfer,postcode,property_type,ppd_category_type,price
0,{666758D6-751F-3363-E053-6B04A8C0D74E},2017-07-07,NW6 5FL,F,A,502500
1,{666758D6-7522-3363-E053-6B04A8C0D74E},2017-12-08,E8 2FA,F,A,534000
2,{666758D6-7523-3363-E053-6B04A8C0D74E},2017-12-08,E8 2FA,F,A,765000
3,{666758D6-7524-3363-E053-6B04A8C0D74E},2017-07-28,NW2 1HJ,F,A,350000
4,{666758D6-7525-3363-E053-6B04A8C0D74E},2017-10-20,TW7 4PX,F,A,313000


# **2. Silver — deduplication (cross-file)**

Deduplication is the one step that must run *after* concat — a re-issued
`transaction_id` could span two files. Keep the latest record per id,
then drop the helper id so the final schema is the five operational
columns in `date → price` order.

In [23]:
dups = df["transaction_id"].duplicated().sum()
print("duplicate transaction_id before dedup:", dups)

df = df.drop_duplicates(subset="transaction_id", keep="last")
print("duplicate transaction_id after dedup:", df["transaction_id"].duplicated().sum())

df = df.drop(columns="transaction_id").reset_index(drop=True)
print("shape after dedup:", df.shape)
df.head()

duplicate transaction_id before dedup: 0
duplicate transaction_id after dedup: 0
shape after dedup: (4437307, 5)


,date_of_transfer,postcode,property_type,ppd_category_type,price
0,2017-07-07,NW6 5FL,F,A,502500
1,2017-12-08,E8 2FA,F,A,534000
2,2017-12-08,E8 2FA,F,A,765000
3,2017-07-28,NW2 1HJ,F,A,350000
4,2017-10-20,TW7 4PX,F,A,313000


# **3. Validation checks**

In [24]:
# Row-count reconciliation vs the total raw rows read
print("raw rows:", raw_row_count)
print("clean rows:", len(df))
print("rows removed:", raw_row_count - len(df))

raw rows: 5295467
clean rows: 4437307
rows removed: 858160


In [25]:
# Null checks
display(df.isnull().sum(), round(df.isnull().mean(), 4))

date_of_transfer     0
postcode             0
property_type        0
ppd_category_type    0
price                0
dtype: int64

date_of_transfer     0.0
postcode             0.0
property_type        0.0
ppd_category_type    0.0
price                0.0
dtype: float64

In [26]:
# Value sanity: only allowed categories, realistic price, expected date range
display(df["property_type"].value_counts())
display(df["ppd_category_type"].value_counts())
print("price min/max:", df["price"].min(), df["price"].max())
print("date range:", df["date_of_transfer"].min(), "->", df["date_of_transfer"].max())
print("column order:", list(df.columns))

property_type
S    1283294
T    1193560
D    1193252
F     767201
Name: count, dtype: int64

ppd_category_type
A    4437307
Name: count, dtype: int64

price min/max: 50000 90000000
date range: 2017-01-01 00:00:00 -> 2021-12-31 00:00:00
column order: ['date_of_transfer', 'postcode', 'property_type', 'ppd_category_type', 'price']


# **4. Export**

In [27]:
os.makedirs("../data/silver", exist_ok=True)
df.to_csv("../data/silver/houseprices_clean.csv", index=False)
print("Wrote ../data/silver/houseprices_clean.csv", df.shape)

Wrote ../data/silver/houseprices_clean.csv (4437307, 5)
